# Testing your best model configurations
This notebook lets you train a new model on the entirety of your training data, and then validate its performance on the test partition. 

### How to use:
Put the directory of your `best_model` folder in the config cell below and run the notebook. The output will be placed in this folder too.

In [ ]:
BEST_RUN_DIR = r"assignment_2\output\05-06-2026--12-49\job_0\best_model"

DEVICE = "cuda"

ADDITIONAL_EPOCHS = 0

From here on, you do not need to alter the code.

In [2]:
import logging
import os
import sys
import torch
import yaml

from jsonschema import validate, ValidationError
from logging.handlers import RotatingFileHandler
from torch import nn

from baseline import Baseline
from custom_logger_formatter import CustomLoggerFormatter
from config.config_validation_template import CONFIG_TEMPLATE
from data import to_dataloaders
from eeg_net import EEGNet
from eeg_net_transformer import EEGNetTransformer
from main import DATASET_MAPPING
from meg_dataset import MEGDataset, LABEL_MAP
from meg_gcnet import MEGGCNet
from meg_gpt import MEGGPT
from train import train, evaluate
from tune import TUNABLE_PARAMS
from visualise import \
    visualise_training, \
    plot_confusion_matrix

os.chdir("..")

c:\Users\joris\miniconda3\envs\INFOMDLR\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialise Logger.

def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("test logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
f_ch = RotatingFileHandler(f"{BEST_RUN_DIR}/testing.log")
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the testing process.")

2026-06-05 13:18:07,601 - test logger - INFO - This file logs the testing process. (69454693.py:24)


In [4]:
# validate the provided config file.
with open(f"{BEST_RUN_DIR}/run_config.yml", 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(f"{BEST_RUN_DIR}/../../config.yml", 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

run = CONFIG["jobs"]["job0"]
for tunable_param in TUNABLE_PARAMS.keys():
    if tunable_param == "dropout":
        run["model_params"][tunable_param] = run["model_params"][tunable_param][0]
    else:
        run[tunable_param] = run[tunable_param][0]

run["n_epochs"] += ADDITIONAL_EPOCHS

logger.info(f"config: {CONFIG}")

2026-06-05 13:18:07,628 - test logger - INFO - config: {'jobs': {'job0': {'dataset': 'cross', 'window_size': 252, 'stride': 203, 'downsample_factor': 2, 'lazy': True, 'train_val_split': [0.8, 0.2], 'batch_size': 107, 'model': 'baseline', 'model_params': {'dropout': 0.17044614691610738}, 'optimiser': 'adam', 'learning_rate': 2.879347852636365e-05, 'weight_decay': 0.049213561767077404, 'n_epochs': 100, 'k_folds': 3, 'tune': True, 'n_trials': 10, 'n_startup_trials': 5, 'min_epochs': 10, 'reduction_factor': 3}}, 'general': {'num_data_workers': 1, 'ommited_sensors': [3, 57, 61, 73, 105, 144, 188, 193, 224, 236]}} (2129972395.py:28)


In [5]:
# Load the data
train_dataset = MEGDataset(
    data_dirs=DATASET_MAPPING[run["dataset"].lower()]["train"],
    window_size=run["window_size"],
    stride=run["stride"],
    ommited_sensors=CONFIG["general"]["ommited_sensors"],
    downsample_factor=run["downsample_factor"],
    lazy=run["lazy"],
)
test_data = MEGDataset(
    data_dirs=DATASET_MAPPING[run["dataset"].lower()]["test"],
    window_size=run["window_size"],
    stride=1,
    ommited_sensors=CONFIG["general"]["ommited_sensors"],
    downsample_factor=run["downsample_factor"],
    lazy=run["lazy"],
)

# Normalise
logger.debug("Fitting normalisation.")
train_dataset.fit_normalisation(list(range(len(train_dataset))))
test_data.mean = train_dataset.mean
test_data.std = train_dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean[:2]={train_dataset.mean[:2]}, std[:2]={train_dataset.std[:2]}"
)

# Val dataset only needed to re-use existing code, not actually used
val_dataset = torch.utils.data.Subset(train_dataset, list(range(0, run["batch_size"])))
logger.debug(f"{len(train_dataset) = }, {len(val_dataset) = }")

# Convert to dataloaders
logger.debug("converting to dataloaders")
train_dataloader, val_dataloader = to_dataloaders(
    [train_dataset, val_dataset], 
    batch_sizes=[run["batch_size"]] * 2, 
    shuffles=[True, False],
    logger=logger,
    num_workers=CONFIG["general"]["num_data_workers"],
    pin_memory=True, # TODO: check if this should be replaced with run["lazy"]
    persistent_workers=True
)

test_dataloader = to_dataloaders(
    [test_data], 
    batch_sizes=[run["batch_size"]], 
    shuffles=[False],
    logger=logger,
    num_workers=0,
    pin_memory=False,
)[0]


2026-06-05 13:18:07,831 - test logger - DEBUG - Fitting normalisation. (1208663785.py:20)
2026-06-05 13:18:12,219 - test logger - DEBUG - Normalisation fitted: mean[:2]=[[ 1.5244903e-12]
 [-1.8072228e-12]], std[:2]=[[2.3489901e-12]
 [3.1855742e-12]] (1208663785.py:24)
2026-06-05 13:18:12,220 - test logger - DEBUG - len(train_dataset) = 8352, len(val_dataset) = 107 (1208663785.py:31)
2026-06-05 13:18:12,221 - test logger - DEBUG - converting to dataloaders (1208663785.py:34)
2026-06-05 13:18:12,222 - test logger - DEBUG - Converting dataset of 8352 elements into DataLoader with 78 partitions of size 107. (data.py:42)
2026-06-05 13:18:12,223 - test logger - DEBUG - Converting dataset of 107 elements into DataLoader with 1 partitions of size 107. (data.py:42)
2026-06-05 13:18:12,224 - test logger - DEBUG - Converting dataset of 842928 elements into DataLoader with 7877 partitions of size 107. (data.py:42)


In [6]:
# Process the model
logger.debug(f"Initialising the model ({run['model']})")
models = {
    "baseline": (
        Baseline, {
            "network_shape": [
                train_dataset.get_n_sensors() * run["window_size"], 
                *(
                    [run["model_params"].get("hidden_size", 64)] * 
                    run["model_params"].get("num_layers", 1)
                ),
                len(LABEL_MAP),
            ],
            "dropout": run["model_params"]["dropout"],
            "logger": logger,
        }
    ),
    "eegnet": (
        EEGNet, {
            "chunk_size": run["window_size"],
            "num_electrodes": train_dataset.get_n_sensors(),
            "num_classes": len(LABEL_MAP),
            "dropout": run["model_params"]["dropout"],
            "logger": logger,
        }
    ),
    "eegnettransformer": (
        EEGNetTransformer, {
            "chunk_size": run["window_size"],
            "num_electrodes": train_dataset.get_n_sensors(),
            "num_classes": len(LABEL_MAP),
            "dropout": run["model_params"]["dropout"],
            "logger": logger,
        }
    ),
    "meggpt": (
        MEGGPT, {
            "num_electrodes": train_dataset.get_n_sensors(),
            "chunk_size": run["window_size"],
            "num_classes": len(LABEL_MAP),
            "logger": logger,
            "d_model": run["model_params"].get("hidden_size", 64),
            "num_heads": run["model_params"].get("num_heads", 4),
            "num_layers": run["model_params"].get("num_layers", 2),
            "patch_size": run["model_params"].get("patch_size", 8),
            "dropout": run["model_params"].get("dropout", 0.1),
        }
    ),
    "meggcnet": (
        MEGGCNet, {
            "num_nodes": train_dataset.get_n_sensors(),
            "in_channels": 1,
            "num_classes": len(LABEL_MAP),
            "temporal_kernel_size": run["model_params"].get("temporal_kernel_size", 3),
            "gamma_learnable": run["model_params"].get("gamma_learnable", True),
            "dropout": run["model_params"].get("dropout", 0.0),
            "logger": logger,
        }
    ),
}
model = None
for name, (cls, kwargs) in models.items():
    if run['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)

# model requirements
logger.debug(f"Initialising the optimiser ({run['optimiser']})")
optimisers = {
    "adam": (torch.optim.Adam, {
        "params": model.parameters(),
        "lr": run["learning_rate"],
        "weight_decay": run["weight_decay"]
    })
}
OPTIMISER = None
for name, (cls, kwargs) in optimisers.items():
    if run['optimiser'].lower() in name:
        OPTIMISER = cls(**kwargs)
        break
assert OPTIMISER is not None, \
    "Provided optimiser in config does not exist."

LOSS_FN = nn.CrossEntropyLoss()

arguments = {
    "model" : model,
    "loss_fn" : LOSS_FN,
    "optimiser": OPTIMISER,
    "n_epochs" : run["n_epochs"],
    "device" : DEVICE,
    "logger" : logger,
    "pruning_callback": None
}

2026-06-05 13:18:12,248 - test logger - DEBUG - Initialising the model (baseline) (629245862.py:2)
2026-06-05 13:18:12,283 - test logger - DEBUG - Model:
Baseline(
  (model): Sequential(
    (0): Linear(in_features=59976, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.17044614691610738, inplace=False)
    (3): Linear(in_features=64, out_features=4, bias=True)
  )
) (629245862.py:69)
2026-06-05 13:18:12,284 - test logger - DEBUG - Total number of parameters: 3,838,788 (629245862.py:70)
2026-06-05 13:18:12,396 - test logger - DEBUG - Initialising the optimiser (adam) (629245862.py:77)


In [7]:
train_losses, train_metrics, val_losses, val_metrics, model = train(
    train_dataloader=train_dataloader, 
    val_dataloader=val_dataloader,
    **arguments
)
train_losses_std, train_metrics_std = None, None
val_losses_std, val_metrics_std = None, None

model.save(BEST_RUN_DIR, prefix="fully_trained_")


Epoch:   0%|          | 0/100 [00:00<?, ?it/s]

2026-06-05 13:18:13,449 - test logger - INFO - -----===== Epoch 0 (training) =====----- (train.py:230)
2026-06-05 13:18:15,815 - test logger - DEBUG - train loss: 2.258298 | accuracy: 0.140187 | [  107/ 8352] (train.py:329)
2026-06-05 13:18:15,989 - test logger - DEBUG - train loss: 0.190045 | accuracy: 0.763240 | [  963/ 8352] (train.py:329)
2026-06-05 13:18:16,195 - test logger - DEBUG - train loss: 0.046485 | accuracy: 0.869678 | [ 1926/ 8352] (train.py:329)
2026-06-05 13:18:16,398 - test logger - DEBUG - train loss: 0.019284 | accuracy: 0.912773 | [ 2889/ 8352] (train.py:329)
2026-06-05 13:18:16,596 - test logger - DEBUG - train loss: 0.013620 | accuracy: 0.932710 | [ 3745/ 8352] (train.py:329)
2026-06-05 13:18:16,806 - test logger - DEBUG - train loss: 0.009643 | accuracy: 0.946049 | [ 4708/ 8352] (train.py:329)
2026-06-05 13:18:17,010 - test logger - DEBUG - train loss: 0.004602 | accuracy: 0.955211 | [ 5671/ 8352] (train.py:329)
2026-06-05 13:18:17,196 - test logger - DEBUG - tr

Epoch:   1%|          | 1/100 [00:06<10:35,  6.42s/it]

2026-06-05 13:18:19,872 - test logger - INFO - -----===== Epoch 1 (training) =====----- (train.py:230)
2026-06-05 13:18:19,934 - test logger - DEBUG - train loss: 0.011285 | accuracy: 1.000000 | [  107/ 8352] (train.py:329)
2026-06-05 13:18:20,128 - test logger - DEBUG - train loss: 0.007300 | accuracy: 1.000000 | [  963/ 8352] (train.py:329)
2026-06-05 13:18:20,324 - test logger - DEBUG - train loss: 0.004434 | accuracy: 1.000000 | [ 1926/ 8352] (train.py:329)
2026-06-05 13:18:20,544 - test logger - DEBUG - train loss: 0.003321 | accuracy: 1.000000 | [ 2889/ 8352] (train.py:329)
2026-06-05 13:18:20,732 - test logger - DEBUG - train loss: 0.004739 | accuracy: 1.000000 | [ 3745/ 8352] (train.py:329)
2026-06-05 13:18:20,942 - test logger - DEBUG - train loss: 0.002028 | accuracy: 1.000000 | [ 4708/ 8352] (train.py:329)
2026-06-05 13:18:21,149 - test logger - DEBUG - train loss: 0.002586 | accuracy: 0.999647 | [ 5671/ 8352] (train.py:329)
2026-06-05 13:18:21,332 - test logger - DEBUG - tr

Epoch:   2%|▏         | 2/100 [00:08<06:09,  3.77s/it]

...

2026-06-05 13:21:27,923 - test logger - INFO - -----===== Epoch 98 (training) =====----- (train.py:230)
2026-06-05 13:21:27,974 - test logger - DEBUG - train loss: 0.003593 | accuracy: 1.000000 | [  107/ 8352] (train.py:329)
2026-06-05 13:21:28,168 - test logger - DEBUG - train loss: 0.002290 | accuracy: 1.000000 | [  963/ 8352] (train.py:329)
2026-06-05 13:21:28,382 - test logger - DEBUG - train loss: 0.003525 | accuracy: 1.000000 | [ 1926/ 8352] (train.py:329)
2026-06-05 13:21:28,595 - test logger - DEBUG - train loss: 0.003563 | accuracy: 1.000000 | [ 2889/ 8352] (train.py:329)
2026-06-05 13:21:28,789 - test logger - DEBUG - train loss: 0.003807 | accuracy: 1.000000 | [ 3745/ 8352] (train.py:329)
2026-06-05 13:21:29,003 - test logger - DEBUG - train loss: 0.004146 | accuracy: 1.000000 | [ 4708/ 8352] (train.py:329)
2026-06-05 13:21:29,220 - test logger - DEBUG - train loss: 0.004577 | accuracy: 1.000000 | [ 5671/ 8352] (train.py:329)
2026-06-05 13:21:29,411 - test logger - DEBUG - t

Epoch:  99%|█████████▉| 99/100 [03:16<00:01,  1.95s/it]

2026-06-05 13:21:29,864 - test logger - INFO - -----===== Epoch 99 (training) =====----- (train.py:230)
2026-06-05 13:21:29,919 - test logger - DEBUG - train loss: 0.003908 | accuracy: 1.000000 | [  107/ 8352] (train.py:329)
2026-06-05 13:21:30,113 - test logger - DEBUG - train loss: 0.002468 | accuracy: 1.000000 | [  963/ 8352] (train.py:329)
2026-06-05 13:21:30,328 - test logger - DEBUG - train loss: 0.002084 | accuracy: 1.000000 | [ 1926/ 8352] (train.py:329)
2026-06-05 13:21:30,544 - test logger - DEBUG - train loss: 0.003195 | accuracy: 1.000000 | [ 2889/ 8352] (train.py:329)
2026-06-05 13:21:30,735 - test logger - DEBUG - train loss: 0.003904 | accuracy: 1.000000 | [ 3745/ 8352] (train.py:329)
2026-06-05 13:21:30,950 - test logger - DEBUG - train loss: 0.002745 | accuracy: 1.000000 | [ 4708/ 8352] (train.py:329)
2026-06-05 13:21:31,166 - test logger - DEBUG - train loss: 0.001879 | accuracy: 1.000000 | [ 5671/ 8352] (train.py:329)
2026-06-05 13:21:31,361 - test logger - DEBUG - t

Epoch: 100%|██████████| 100/100 [03:18<00:00,  1.98s/it]

2026-06-05 13:21:31,817 - test logger - INFO - Done training (train.py:261)
2026-06-05 13:21:31,818 - test logger - INFO - Saving model to assignment_2\output\05-06-2026--12-49\job_0\best_model/fully_trained_best_Baseline.pth... (base_model.py:44)


In [8]:
visualise_training(
    train_losses, 
    train_metrics,
    val_losses, 
    val_metrics,
    BEST_RUN_DIR + "/fully_trained_",
)

In [ ]:
train_accuracy, train_predictions, train_targets = evaluate(
    dataloader=train_dataloader, 
    model=model,
    device=DEVICE,
    logger=logger,
)
logger.critical(f"Test accuracy: {train_accuracy}")

plot_confusion_matrix(
    train_targets,
    train_predictions,
    LABEL_MAP.keys(),
    None,
    "train",
    BEST_RUN_DIR,
    logger
)

Evaluating batches: 100%|██████████| 79/79 [00:01<00:00, 42.97it/s]


2026-06-05 13:21:34,063 - test logger - INFO - Saving confusion matrix to assignment_2\output\05-06-2026--12-49\job_0\best_modeltrain_Confusion_Matrix.png (visualise.py:235)


In [10]:
logger.info("Evaluating on test set.")
test_accuracy, test_predictions, test_targets = evaluate(
    dataloader=test_dataloader, 
    model=model,
    device=DEVICE,
    logger=logger,
)
logger.critical(f"Test accuracy: {test_accuracy}")

plot_confusion_matrix(
    test_targets,
    test_predictions,
    LABEL_MAP.keys(),
    None,
    "test",
    BEST_RUN_DIR,
    logger
)

2026-06-05 13:21:34,070 - test logger - INFO - Evaluating on test set. (2330992771.py:1)


Evaluating batches: 100%|██████████| 7878/7878 [02:46<00:00, 47.30it/s]


2026-06-05 13:24:20,663 - test logger - CRITICAL - Test accuracy: (0.5743728876113892,) (2330992771.py:8)
2026-06-05 13:24:20,855 - test logger - INFO - Saving confusion matrix to assignment_2\output\05-06-2026--12-49\job_0\best_modeltest_Confusion_Matrix.png (visualise.py:235)


In [11]:
with open(f"{BEST_RUN_DIR}\\metrics.md", "w") as file:
    file.write("# Testing metrics\n")
    file.write("training (on entire dataset):\n")
    file.write(f"accuracy: {train_accuracy}\n")
    file.write("testing:\n")
    file.write(f"accuracy: {test_accuracy}\n")